# Safety Classifier — Fine-tuning

**Before running anything:**
1. `Runtime → Change runtime type → T4 GPU`
2. Run cells top to bottom
3. For parts 2–4: only re-run the **Config** cell (change `PART`) then jump straight to the **Train** cell

## 1 · Install dependencies

In [ ]:
!pip install transformers datasets scikit-learn accelerate -q

## 2 · GPU check

In [ ]:
import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected — go to Runtime > Change runtime type and select T4 GPU")

## 3 · Hugging Face login

Required to access `mental/mental-roberta-base` (gated model).  
Make sure you've accepted the model terms at https://huggingface.co/mental/mental-roberta-base first.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 4 · Mount Google Drive

Checkpoints are saved to Drive so they survive if the session dies.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/pfa_classifier'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Checkpoints will be saved to: {DRIVE_DIR}")

## 5 · Config

**Change `PART` here before each run (1 → 2 → 3 → 4), then jump to the Train cell.**

In [ ]:
# ── Change this before each run: 1 → 2 → 3 → 4 ──────────────────────────────
PART = 1
# ─────────────────────────────────────────────────────────────────────────────

TOTAL_PARTS  = 4
BASE_MODEL   = "mental/mental-roberta-base"
DATASET_NAME = "vibhorag101/suicide_prediction_dataset_phr"
MAX_LENGTH   = 512
BATCH_SIZE   = 16
FREEZE_LAYERS = 8
CRISIS_WEIGHT = 3.0

MODEL_NAME = BASE_MODEL if PART == 1 else f"{DRIVE_DIR}/part_{PART - 1}"
OUTPUT_DIR = f"{DRIVE_DIR}/part_{PART}"

print(f"Part {PART}/{TOTAL_PARTS}")
print(f"Loading from : {MODEL_NAME}")
print(f"Saving to    : {OUTPUT_DIR}")

## 6 · Setup (imports, tokenizer, dataset)

Run once — no need to re-run between parts.

In [ ]:
import numpy as np
from datasets import load_dataset
from sklearn.metrics import classification_report, recall_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def preprocess(batch):
    tokens = tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)
    tokens["labels"] = [1 if l == "suicide" else 0 for l in batch["label"]]
    return tokens

raw   = load_dataset(DATASET_NAME)
split = raw["train"].train_test_split(test_size=0.1, seed=42)
dataset = {"train": split["train"], "validation": split["test"], "test": raw["test"]}

remove_cols = dataset["train"].column_names
tokenized = {
    s: ds.map(preprocess, batched=True, remove_columns=remove_cols)
    for s, ds in dataset.items()
}
collator = DataCollatorWithPadding(tokenizer)
print("Dataset ready.")
print(f"  Train: {len(tokenized['train'])} | Val: {len(tokenized['validation'])} | Test: {len(tokenized['test'])}")

## 7 · Train

For parts 2–4: update the **Config** cell, re-run it, then run this cell.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    crisis_recall = recall_score(labels, preds, pos_label=1)
    print(classification_report(labels, preds, target_names=["safe", "crisis"]))
    return {"crisis_recall": crisis_recall}

class WeightedLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        weights = torch.tensor([1.0, CRISIS_WEIGHT], device=outputs.logits.device)
        loss = torch.nn.functional.cross_entropy(outputs.logits, labels, weight=weights)
        return (loss, outputs) if return_outputs else loss

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "safe", 1: "crisis"},
    label2id={"safe": 0, "crisis": 1},
    ignore_mismatched_sizes=True,
)
if PART == 1:
    for i, layer in enumerate(model.roberta.encoder.layer):
        if i < FREEZE_LAYERS:
            for param in layer.parameters():
                param.requires_grad = False

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    report_to="none",
    save_total_limit=1,
    save_only_model=True,
)

trainer = WeightedLossTrainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()

if PART == TOTAL_PARTS:
    print("\n--- Final test set evaluation ---")
    trainer.evaluate(tokenized["test"])

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\nPart {PART} saved to {OUTPUT_DIR}")
if PART < TOTAL_PARTS:
    print(f"Next: set PART = {PART + 1} in the Config cell and re-run Config + Train.")
else:
    print("Training complete. Download part_4 from Drive.")